<a href="https://colab.research.google.com/github/Munjiwon/SpecialTopics-in-TextMining/blob/master/ch03/sgns_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3주차 실습 2 — PyTorch 로 Skip-gram + Negative Sampling 직접 구현

**이 노트북의 새 개념**: 소프트맥스 대신 "이 (중심, 문맥) 쌍이 진짜인가"를 묻는 이진 분류로 학습한다.

J = log σ(u_o·v_c) + Σ_k log σ(−u_k·v_c),  부정 표본 k 는 P_n(w) ∝ U(w)^{3/4} 에서 뽑는다.

런타임 → 런타임 유형 변경 → T4 GPU 로 두면 빨라지지만, 이 규모는 CPU 에서도 몇 분이면 끝난다.

In [1]:
%pip install -q kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 100.9 MB/s eta 0:00:00


In [2]:
import sys, torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

Python 3.13.15 | torch 2.11.0+cu128 | GPU: True


In [3]:
# 말뭉치 준비와 형태소 토큰화 — 1주차와 같은 Steam 리뷰 감성 말뭉치("레이블<TAB>텍스트")를 내려받고,
# 단어 임베딩 학습에는 문장만 쓰므로(레이블은 따로 보관만 한다) 한국어는 형태소 단위로 나눈다(2주차에서 본 것처럼 형태소 단위가 유리하다).
STEAM_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/steam.txt"
CORPUS_PATH = "data/steam.txt"  # code03/data/ 에 저장 (노트북 위치 기준 상대경로)
USE_OWN_CORPUS = False
if USE_OWN_CORPUS:
    from google.colab import files
    uploaded = files.upload()
    CORPUS_PATH = "/content/" + next(iter(uploaded))
else:
    import os, urllib.request
    os.makedirs(os.path.dirname(CORPUS_PATH), exist_ok=True)
    if not os.path.exists(CORPUS_PATH):  # 이미 내려받았으면 건너뜀
        urllib.request.urlretrieve(STEAM_URL, CORPUS_PATH)

# 파일 형식: 한 줄에 리뷰 하나, "레이블(0=부정, 1=긍정)<TAB>리뷰 문장"
lines, labels = [], []
with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        label, text = line.rstrip("\n").split("\t", 1)   # 첫 탭을 기준으로 두 칸으로 나눔
        if text.strip():
            labels.append(int(label))
            lines.append(text.strip())
print(f"리뷰 {len(lines):,}개 (부정 {labels.count(0):,} / 긍정 {labels.count(1):,})")

from kiwipiepy import Kiwi
kiwi = Kiwi()
# 문장(줄) 하나를 토큰 목록 하나로 — gensim 과 직접 구현 모두 이 형태를 쓴다
sentences = [[t.form for t in kiwi.tokenize(l)] for l in lines]
print(f"문장 {len(sentences):,}개, 토큰 {sum(map(len, sentences)):,}개")

리뷰 100,000개 (부정 50,004 / 긍정 49,996)
문장 100,000개, 토큰 2,548,479개


## 어휘와 학습 쌍 만들기

In [4]:
from collections import Counter
import random
random.seed(0); torch.manual_seed(0)

counts = Counter(w for s in sentences for w in s)
vocab = [w for w, c in counts.items() if c >= 3]
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
ids = [[w2i[w] for w in s if w in w2i] for s in sentences]

WINDOW = 3
pairs = [(s[i], s[j]) for s in ids for i in range(len(s))
         for j in range(max(0, i - WINDOW), min(len(s), i + WINDOW + 1)) if j != i]
print(f"어휘 {V:,}개, (중심, 문맥) 쌍 {len(pairs):,}개")

# 잡음 분포 P_n(w) ∝ U(w)^{3/4}
unigram = torch.tensor([counts[w] for w in vocab], dtype=torch.float)
noise = unigram ** 0.75
noise /= noise.sum()

어휘 15,492개, (중심, 문맥) 쌍 13,876,796개


## 모델 — 임베딩 두 개(중심용 v, 문맥용 u)

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class SGNS(nn.Module):
    def __init__(self, V, dim):
        super().__init__()
        self.v = nn.Embedding(V, dim)          # 중심 단어 벡터
        self.u = nn.Embedding(V, dim)          # 문맥 단어 벡터
        nn.init.uniform_(self.v.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.u.weight)

    def forward(self, center, context, negatives):
        vc = self.v(center)                                   # (B, d)
        uo = self.u(context)                                  # (B, d)
        uk = self.u(negatives)                                # (B, K, d)
        pos = F.logsigmoid((uo * vc).sum(-1))                 # log σ(u_o·v_c)
        neg = F.logsigmoid(-(uk @ vc.unsqueeze(-1)).squeeze(-1)).sum(-1)   # Σ log σ(−u_k·v_c)
        return -(pos + neg).mean()                            # 최대화 → 음수를 최소화

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SGNS(V, dim=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)

## 학습

In [6]:
K, BATCH, EPOCHS = 5, 512, 3
data = torch.tensor(pairs)
for epoch in range(EPOCHS):
    perm = torch.randperm(len(data))
    total = 0.0
    for b in range(0, len(data), BATCH):
        batch = data[perm[b:b + BATCH]]
        center, context = batch[:, 0].to(device), batch[:, 1].to(device)
        negatives = torch.multinomial(noise, len(batch) * K, replacement=True).view(len(batch), K).to(device)
        loss = model(center, context, negatives)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(batch)
    print(f"epoch {epoch + 1}: 평균 손실 {total / len(data):.4f}")

epoch 1: 평균 손실 2.4791
epoch 2: 평균 손실 2.4684
epoch 3: 평균 손실 2.4643


## 최근접 이웃 확인

In [7]:
emb = F.normalize(model.v.weight.detach().cpu(), dim=1)
def neighbors(word, k=5):
    sims = emb @ emb[w2i[word]]
    return [vocab[i] for i in sims.topk(k + 1).indices.tolist() if vocab[i] != word][:k]

nouns = Counter(t.form for l in lines for t in kiwi.tokenize(l) if t.tag == "NNG" and len(t.form) > 1)
for q in [w for w, _ in nouns.most_common(40) if w in w2i][:3]:    # 자주 나오는 일반명사 3개
    print(q, "→", neighbors(q))

게임 → ['액션', '퍼즐', '느낌', '소재', '디펜스']
플레이 → ['플탐', '유추', '타임', '려면', '위주']
재미 → ['어이', '몰입도', '화염', '쓸모', '수']


## 직접 해 보기
1. 부정 표본 수 K 를 1, 5, 20 으로 바꿔 손실과 이웃이 어떻게 달라지는지 보자.
2. 잡음 분포의 지수 0.75 를 1.0 으로 바꾸면 무엇이 달라지는가(`negative_sampling_dist.ipynb` 참고)?
3. 학습 전(초기화 직후)의 이웃과 비교해 학습이 실제로 무엇을 바꿨는지 확인하자.